# 07 — Real-Time Demo: Detection, Cumulative Counts & Minigame Odds

Interactive live demo of the best detector + tracker combination.
For each frame it shows:

1. **Annotated video frame** — bounding boxes with class labels and track IDs (every frame)
2. **Cumulative stat count bar chart** — how many times each stat icon has been seen (every 30 frames)
3. **Minigame posterior bar chart** — evolving probability of each end-of-round minigame (every 30 frames)

Press `q` or `Esc` to quit. Set `SAVE_OUTPUT = True` to write an annotated `.mp4`.

Prerequisites: `02_train_yolo.ipynb` · `04_tracker_architecture_botsort.ipynb` · `04b_tracker_architecture_bytetrack.ipynb`

## 0 — Detector + Tracker Selection

**Edit only this cell** to choose which detector and tracker to use.

| Setting | Options |
|---|---|
| `ACTIVE_DETECTOR` | `'yolo'` · `'rtdetr'` |
| `ACTIVE_TRACKER` | `'botsort'` · `'bytetrack'` |

The correct arch config JSON is loaded automatically from `configs/` based on your choice:
- `botsort` → `configs/kartector_botsort_arch.json` (ReID Option B, match_thresh=0.99)
- `bytetrack` → `configs/kartector_bytetrack_arch.json` (no ReID, match_thresh=0.99)

In [12]:
# ════════════════════════════════════════════════════════════════════════
# ▶  EDIT HERE — choose detector and tracker
# ════════════════════════════════════════════════════════════════════════
ACTIVE_DETECTOR = 'yolo'      # 'yolo' | 'rtdetr'
ACTIVE_TRACKER  = 'bytetrack'   # 'botsort' | 'bytetrack'
# ════════════════════════════════════════════════════════════════════════

## 0b — Configuration (auto-loaded from arch JSON)

Loads `configs/kartector_<tracker>_arch.json` based on the selection above and
derives all runtime settings (weights, tracker yaml, conf, IOU, ReID params).

If `ACTIVE_DETECTOR = 'rtdetr'`, the YOLO weights from the arch JSON are
overridden with `runs/rtdetr/final/weights/best.pt`.

In [13]:
from pathlib import Path
import json, cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("TkAgg")   # change to "Qt5Agg" or "MacOSX" if TkAgg unavailable
from ultralytics import YOLO
import pandas as pd

REPO_ROOT = Path("../")
RUNS_DIR  = (REPO_ROOT / "runs").resolve()

# ── Select arch config based on ACTIVE_TRACKER ──────────────────────────────────────────
_ARCH_CFG_MAP = {
    'botsort':   REPO_ROOT / 'configs' / 'kartector_botsort_arch.json',
    'bytetrack': REPO_ROOT / 'configs' / 'kartector_bytetrack_arch.json',
}
_ARCH_CFG_PATH = _ARCH_CFG_MAP.get(ACTIVE_TRACKER,
                                    REPO_ROOT / 'configs' / 'kartector_botsort_arch.json')

if _ARCH_CFG_PATH.exists():
    _arch_cfg     = json.loads(_ARCH_CFG_PATH.read_text())
    WEIGHTS       = REPO_ROOT / _arch_cfg.get('weights', 'runs/yolo26/final/weights/best.pt')
    TRACKER_CFG   = str(REPO_ROOT / _arch_cfg.get('tracker_yaml', 'configs/kartector_botsort_reentry.yaml'))
    CONF          = _arch_cfg.get('conf', 0.25)
    IOU           = _arch_cfg.get('iou', 0.70)
    MAX_GAP       = _arch_cfg.get('max_gap', 45)
    MAX_DIST      = _arch_cfg.get('max_dist_pct', 40.0)
    REID_OPTION   = _arch_cfg.get('reid_option', 'none')
    REID_B_PARAMS = _arch_cfg.get('reid_b_params', {})
    print(f"Loaded arch config : {_ARCH_CFG_PATH.name}")
else:
    print(f"WARNING: arch config not found at {_ARCH_CFG_PATH}, using defaults")
    WEIGHTS       = RUNS_DIR / 'yolo26' / 'final' / 'weights' / 'best.pt'
    TRACKER_CFG   = str(REPO_ROOT / 'configs' / 'kartector_botsort_reentry.yaml')
    CONF          = 0.25
    IOU           = 0.70
    MAX_GAP       = 45
    MAX_DIST      = 40.0
    REID_OPTION   = 'none'
    REID_B_PARAMS = {}

# ── Override weights for RT-DETR ───────────────────────────────────────────────────────────────────────────
_RTDETR_WEIGHTS = RUNS_DIR / 'rtdetr' / 'final' / 'weights' / 'best.pt'
if ACTIVE_DETECTOR == 'rtdetr':
    if _RTDETR_WEIGHTS.exists():
        WEIGHTS = _RTDETR_WEIGHTS
    else:
        print(f"WARNING: RT-DETR weights not found at {_RTDETR_WEIGHTS}, falling back to YOLO")
        ACTIVE_DETECTOR = 'yolo'

# ── Validate tracker config file ──────────────────────────────────────────────────────────────────────────────────────
if not Path(TRACKER_CFG).exists():
    _fallback = 'bytetrack.yaml' if ACTIVE_TRACKER == 'bytetrack' else 'botsort.yaml'
    print(f"WARNING: tracker config not found at {TRACKER_CFG}, falling back to {_fallback}")
    TRACKER_CFG = _fallback

# ── ReID mode: bytetrack never uses ReID ─────────────────────────────────────────────────────────────────────────────
_ACTIVE_REID = 'none' if ACTIVE_TRACKER == 'bytetrack' else REID_OPTION

# ── Input video ──────────────────────────────────────────────────────────────────────────────────────────────────────────
CHUNKS_DIR  = REPO_ROOT / "data/processed/labelstudiochunks"
demo_videos = sorted(CHUNKS_DIR.glob("*.mp4"))
VIDEO_PATH  = str(demo_videos[10]) if demo_videos else 0

with open(REPO_ROOT / "data/splits/splits.json") as f:
    splits = json.load(f)
CLASSES   = splits["classes"]
N_CLASSES = len(CLASSES)

# ── Compute device ───────────────────────────────────────────────────────────────────────────────────────────────────────────
import torch as _torch_cfg
_DEVICE = _torch_cfg.device(
    'cuda' if _torch_cfg.cuda.is_available() else
    'mps'  if _torch_cfg.backends.mps.is_available() else 'cpu')

# ── Boxmot tracker factory (used by live and offline loops) ────────────────────────
def _make_boxmot_tracker():
    """Instantiate a fresh BotSort tracker using arch config ReID params."""
    from boxmot.trackers.botsort.botsort import BotSort
    from boxmot.utils import WEIGHTS as _BW
    return BotSort(
        reid_weights=_BW / 'osnet_x0_25_msmt17.pt',
        device=_DEVICE, half=False,
        track_buffer=REID_B_PARAMS.get('track_buffer', 30),
        match_thresh=REID_B_PARAMS.get('match_thresh', 0.70),
        appearance_thresh=REID_B_PARAMS.get('appearance_thresh', 0.40),
        proximity_thresh=REID_B_PARAMS.get('proximity_thresh', 0.5),
        with_reid=True,
    )

print(f"Compute device  : {_DEVICE}")
print(f"Active detector : {ACTIVE_DETECTOR}  ->  {WEIGHTS}")
print(f"Active tracker  : {ACTIVE_TRACKER}  ->  {TRACKER_CFG}")
print(f"ReID mode       : {_ACTIVE_REID}")
print(f"CONF={CONF}  IOU={IOU}  MAX_GAP={MAX_GAP}  MAX_DIST={MAX_DIST}")
print(f"Video           : {VIDEO_PATH}")

Loaded arch config : kartector_bytetrack_arch.json
Compute device  : mps
Active detector : yolo  ->  ../runs/yolo26/final/weights/best.pt
Active tracker  : bytetrack  ->  ../configs/kartector_bytetrack_best.yaml
ReID mode       : none
CONF=0.4  IOU=0.7  MAX_GAP=90  MAX_DIST=70.0
Video           : ../data/processed/labelstudiochunks/AY_G1_chunk_0010.mp4


## 1 — Minigame Model

Bayesian minigame posterior model. `compute_posterior` is imported from `helpers.py`.

In [14]:
from helpers import compute_posterior

# ── Minigame names ─────────────────────────────────────────────────────────
# TODO: Replace with the actual minigame names
MINIGAMES = [
    "Single Race",
    "Drag Race",
    "Destruction Derby",
]

# ── Prior over minigames (uniform until you have better data) ──────────────
PRIOR = np.ones(len(MINIGAMES)) / len(MINIGAMES)

# ── Likelihood: P(stat_class | minigame) ──────────────────────────────────
# Shape: (n_minigames, n_stat_classes)
# Each row must sum to 1.
# TODO: Fill in the actual drop-rate distributions per minigame.
LIKELIHOODS = np.array([
    # Boost  Charge  Defense  Glide   HP    Offense  TopSpeed  Turn    Weight
    [0.083,0.083,  0.083,   0.083,  0.075,  0.062,   0.083,    0.083,  0.083],  # Single Race
    [0.082,0.082,  0.082,   0.082,  0.074,  0.082,   0.082,    0.082,  0.074],  # Drag Race
    [0.09, 0.072,  0.108,   0.072,  0.072,  0.129,    0.05,    0.072,  0.072],  # Destruction Derby
])

# Normalise each row so it sums to exactly 1
LIKELIHOODS = LIKELIHOODS / LIKELIHOODS.sum(axis=1, keepdims=True)

# Sanity check
assert LIKELIHOODS.shape == (len(MINIGAMES), N_CLASSES), "Shape mismatch"
assert np.allclose(LIKELIHOODS.sum(axis=1), 1.0), "Rows must sum to 1"

print("Minigame model ready.")

Minigame model ready.


## 2 — Real-Time Demo Loop

Opens a three-panel matplotlib window:
- **Left**: annotated video frame (updated every frame) with bounding boxes + track IDs
- **Centre**: cumulative stat-count bar chart (updated every 30 frames)
- **Right**: minigame posterior bar chart (updated every 30 frames)

The video panel updates every frame for smooth playback; the charts update every 30 frames
to reduce matplotlib rendering overhead without losing meaningful resolution.

A rolling merge pass (`_rolling_merge`) runs every `MERGE_EVERY` frames to cancel duplicate
counts from re-identified tracks, and a final merge pass with `MERGE_LAG=0` runs after
the last frame to catch any tracks still warm at the end.

Press `q` or `Esc` to quit. Set `SAVE_OUTPUT = True` to write an annotated `.mp4`.

In [15]:
SAVE_OUTPUT   = False
OUTPUT_PATH   = REPO_ROOT / "runs" / "demo_output.mp4"

# ── Rolling merge config ───────────────────────────────────────────────────
# Tracks that ended more than MERGE_LAG frames ago are eligible to be
# stitched with newer tracks. Every MERGE_EVERY frames we run a rolling
# merge on the cold portion of the track history and cancel any duplicate
# counts that the merge reveals.
MERGE_LAG   = 90    # frames a track must be gone before it's eligible (>= max_gap)
MERGE_EVERY = 15    # how often (frames) to run the rolling merge
MAX_DIST_PCT = 70.0 # spatial threshold (% of frame diagonal) for stitching

# ── Colour palette ─────────────────────────────────────────────────────────
PALETTE = [
    (255, 56,  56),  (255, 157, 151), (255, 112,  31), (255, 178,  29),
    (207, 210,  49), (72,  249,  10), (146, 204,  23), (61,  219, 134),
    (26,  147,  52),
]
def cls_color(cid):
    return PALETTE[int(cid) % len(PALETTE)]

# ── Build matplotlib figure (updated each frame) ───────────────────────────
fig = plt.figure(figsize=(18, 6), constrained_layout=True)
gs  = fig.add_gridspec(1, 3)
ax_vid   = fig.add_subplot(gs[0, 0])
ax_count = fig.add_subplot(gs[0, 1])
ax_odds  = fig.add_subplot(gs[0, 2])
plt.ion()

def update_figure(frame_bgr, counts, posterior, charts_only=False):
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

    # Panel 1 — video frame (always updated)
    ax_vid.clear()
    ax_vid.imshow(frame_rgb)
    ax_vid.set_title("Live Detections", fontsize=10)
    ax_vid.axis("off")

    if not charts_only:
        # Panel 2 — cumulative counts
        ax_count.clear()
        colors_c = [tuple(c/255 for c in cls_color(i)) for i in range(N_CLASSES)]
        bars = ax_count.bar(CLASSES, counts, color=colors_c, edgecolor="white")
        for bar, v in zip(bars, counts):
            if v > 0:
                ax_count.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                              str(int(v)), ha="center", fontsize=8, fontweight="bold")
        ax_count.set_title("Cumulative Stat Counts", fontsize=10)
        ax_count.set_ylabel("Count")
        plt.setp(ax_count.get_xticklabels(), rotation=30, ha="right", fontsize=8)
        ax_count.grid(axis="y", alpha=0.3)

        # Panel 3 — minigame posterior
        ax_odds.clear()
        colors_m = plt.cm.Set2.colors
        bars2 = ax_odds.bar(MINIGAMES, posterior,
                            color=[colors_m[i % len(colors_m)] for i in range(len(MINIGAMES))],
                            edgecolor="white")
        for bar, p in zip(bars2, posterior):
            ax_odds.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                         f"{p:.1%}", ha="center", fontsize=8, fontweight="bold")
        ax_odds.set_ylim(0, 1.05)
        ax_odds.set_ylabel("Probability")
        ax_odds.set_title("Minigame Odds", fontsize=10)
        ax_odds.grid(axis="y", alpha=0.3)

    fig.canvas.draw()
    fig.canvas.flush_events()



# ── Main tracking loop ─────────────────────────────────────────────────────
_det_model  = YOLO(str(WEIGHTS))
_det_model.to(_DEVICE)
cap         = cv2.VideoCapture(VIDEO_PATH)
fps         = cap.get(cv2.CAP_PROP_FPS) or 30
frame_w     = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_h     = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()

writer   = None
if SAVE_OUTPUT:
    OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    writer = cv2.VideoWriter(str(OUTPUT_PATH),
                             cv2.VideoWriter_fourcc(*"mp4v"),
                             fps, (fig.canvas.get_width_height()))

counts     = np.zeros(N_CLASSES)
posterior  = compute_posterior(counts, PRIOR, LIKELIHOODS)
seen_tids  = set()   # set of (tid, cid) that have already been counted
frame_idx  = 0

# ── Rolling-merge state ────────────────────────────────────────────────────
# track_history[tid] = {'cid', 'last_frame', 'last_xy', 'first_xy', 'counted'}
track_history = {}

def _update_track_history(tid, cid, x1, y1, x2, y2):
    cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
    if tid not in track_history:
        track_history[tid] = {'cid': cid, 'first_frame': frame_idx,
                               'first_xy': (cx, cy), 'last_frame': frame_idx,
                               'last_xy': (cx, cy), 'counted': False}
    else:
        track_history[tid]['last_frame'] = frame_idx
        track_history[tid]['last_xy']    = (cx, cy)

def _rolling_merge(frame_w, frame_h):
    """Stitch cold tracks to newer tracks; cancel duplicate counts."""
    global counts
    diag = (frame_w**2 + frame_h**2) ** 0.5
    cold_cutoff = frame_idx - MERGE_LAG

    cold_tids = [t for t, h in track_history.items()
                 if h['last_frame'] < cold_cutoff]
    warm_tids = [t for t, h in track_history.items()
                 if h['last_frame'] >= cold_cutoff]

    for cold in cold_tids:
        ch = track_history[cold]
        if not ch['counted']: continue   # never counted → nothing to cancel
        for warm in warm_tids:
            wh = track_history[warm]
            if wh['cid'] != ch['cid']:   continue   # different class
            if wh.get('merged'):         continue   # warm already merged away
            if not wh['counted']:        continue   # warm never counted, nothing to cancel
            if wh['first_frame'] <= ch['last_frame']: continue  # overlapping
            gap = wh['first_frame'] - ch['last_frame']
            if gap > MERGE_LAG:          continue   # too far apart in time
            dx = ch['last_xy'][0] - wh['first_xy'][0]
            dy = ch['last_xy'][1] - wh['first_xy'][1]
            if ((dx**2 + dy**2)**0.5) / diag * 100 <= MAX_DIST_PCT:
                # Stitch: warm is a continuation of cold — cancel its count
                wh['merged'] = True     # mark so we don't double-cancel
                counts[wh['cid']] = max(0, counts[wh['cid']] - 1)
                seen_tids.discard((warm, wh['cid']))
                break   # each cold track stitches to at most one warm track

def _process_tracks_07(frame_bgr, tracks, boxes_xyxy=None):
    """Draw boxes, update history, and count new unique stat icons."""
    global counts, seen_tids
    if tracks is None: return
    for t in tracks:
        if boxes_xyxy is not None:
            x1,y1,x2,y2 = map(int, t[:4])
            tid, cid = int(t[4]), int(t[6]) if len(t) > 6 else 0
            c = float(t[5])
        else:
            x1,y1,x2,y2 = map(int, t[0])
            tid, cid, c = int(t[1]), int(t[2]), float(t[3])
        if cid < 0 or cid >= N_CLASSES: continue
        color = cls_color(cid)
        cv2.rectangle(frame_bgr, (x1,y1), (x2,y2), color, 2)
        cv2.putText(frame_bgr, f"{CLASSES[cid]} T{tid} {c:.2f}", (x1, y1-6),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1)
        _update_track_history(tid, cid, x1, y1, x2, y2)
        _key = (tid, cid)
        if _key not in seen_tids:
            counts[cid] += 1
            seen_tids.add(_key)
            track_history[tid]['counted'] = True

try:
    if _ACTIVE_REID == 'B':
        import numpy as _np07
        _tracker07 = _make_boxmot_tracker()
        _cap07 = cv2.VideoCapture(VIDEO_PATH)
        while True:
            _ret, frame_bgr = _cap07.read()
            if not _ret: break
            _res = _det_model(frame_bgr, conf=CONF, verbose=False)
            _dets = _res[0].boxes
            if _dets is not None and len(_dets):
                _darr = _np07.concatenate([
                    _dets.xyxy.cpu().numpy(),
                    _dets.conf.cpu().numpy().reshape(-1,1),
                    _dets.cls.cpu().numpy().reshape(-1,1)], axis=1)
            else:
                _darr = _np07.empty((0, 6))
            _tracks = _tracker07.update(_darr, frame_bgr)
            _process_tracks_07(frame_bgr, _tracks, boxes_xyxy=True)
            if frame_idx % MERGE_EVERY == 0:
                _rolling_merge(frame_bgr.shape[1], frame_bgr.shape[0])
            posterior = compute_posterior(counts, PRIOR, LIKELIHOODS)
            if writer is not None:
                writer.write(frame_bgr)
            update_figure(frame_bgr, counts, posterior,
                          charts_only=(frame_idx % 30 != 0))
            fig.canvas.manager.set_window_title(
                f"KARTector Demo — frame {frame_idx}  |  q to quit")
            frame_idx += 1
            if plt.waitforbuttonpress(timeout=0.001) is not None: break
        # ── Final merge pass: treat all tracks as cold ────────────────
        _saved_lag, MERGE_LAG = MERGE_LAG, 0
        _rolling_merge(frame_bgr.shape[1], frame_bgr.shape[0])
        MERGE_LAG = _saved_lag
        _cap07.release()
    else:
        for result in _det_model.track(VIDEO_PATH, tracker=TRACKER_CFG,
                                       conf=CONF, iou=IOU, stream=True, verbose=False):
            frame_bgr = result.orig_img.copy()
            if result.boxes is not None and result.boxes.id is not None:
                tracks_ul = list(zip(
                    result.boxes.xyxy.cpu().numpy(),
                    result.boxes.id.cpu().numpy().astype(int),
                    result.boxes.cls.cpu().numpy().astype(int),
                    result.boxes.conf.cpu().numpy()))
                _process_tracks_07(frame_bgr, tracks_ul, boxes_xyxy=None)
            if frame_idx % MERGE_EVERY == 0:
                _rolling_merge(frame_bgr.shape[1], frame_bgr.shape[0])
            posterior = compute_posterior(counts, PRIOR, LIKELIHOODS)
            if writer is not None:
                writer.write(frame_bgr)
            update_figure(frame_bgr, counts, posterior,
                          charts_only=(frame_idx % 30 != 0))
            fig.canvas.manager.set_window_title(
                f"KARTector Demo — frame {frame_idx}  |  q to quit")
            frame_idx += 1
            if plt.waitforbuttonpress(timeout=0.001) is not None: break
        # ── Final merge pass: treat all tracks as cold ────────────────
        _saved_lag, MERGE_LAG = MERGE_LAG, 0
        _rolling_merge(frame_bgr.shape[1], frame_bgr.shape[0])
        MERGE_LAG = _saved_lag

except KeyboardInterrupt:
    pass

if writer:
    writer.release()
plt.ioff()
print(f"Demo finished at frame {frame_idx}. Close the window to continue.")
plt.show(block=True)
plt.close(fig)
print(f"Demo finished at frame {frame_idx}.")
print("Final stat counts:")
for cname, cnt in zip(CLASSES, counts):
    print(f"  {cname:<12} {int(cnt)}")
print("\nFinal posterior:")
for mg, p in zip(MINIGAMES, posterior):
    print(f"  {mg:<15} {p:.2%}")

Demo finished at frame 489. Close the window to continue.
Demo finished at frame 489.
Final stat counts:
  Boost        2
  Charge       3
  Defense      0
  Glide        0
  HP           0
  Offense      1
  Top Speed    2
  Turn         0
  Weight       2

Final posterior:
  Single Race     44.46%
  Drag Race       40.61%
  Destruction Derby 14.93%
